# `HostBase`

`nematics3d.HostBase` extends the structured object model of `ClassBase` with a reactive options object and a managed update pipeline. In a typical `HostBase` descendant, users do not edit arbitrary internal state. They modify a declared host input, a field in `host.opts`, or a writable property, and the object routes that change through `act_commit()` so validation, recomputation, synchronization, and wrapper forwarding can happen in one controlled place.

This tutorial is a reference-oriented guide to that public interface. It assumes the naming and inspection conventions from the `ClassBase` reference tutorial. The most important new idea is that a `HostBase` object is **reactive**: changing a public input can immediately update dependent state.


## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** The following cell defines a small `HostBase` subclass and its paired `OptsBase`. The example is deliberately simple so that the host/opts update behavior can be seen without unrelated scientific code.


In [ ]:
from dataclasses import dataclass

from nematics3d.core.class_base import AttrDef
from nematics3d.core.host_base import HostBase, OptsBase


@dataclass(slots=True, repr=False)
class ExampleOpts(OptsBase):
    scale: float = 1.0
    offset: float = 0.0

    __attrs__ = {
        **OptsBase.__attrs__,
        "scale": "Multiplicative scale applied to the raw value.",
        "offset": "Additive offset applied after scaling.",
    }

    impl_defaults_frozen = {
        **OptsBase.impl_defaults_frozen,
        "scale": 1.0,
        "offset": 0.0,
    }


class ExampleHost(HostBase):
    __attr_defs__ = {
        "raw_value": AttrDef(
            "Primary numeric input.",
            kind="raw",
            is_reapply_opts_after_raw=True,
        ),
        "calc_result": AttrDef(
            "Result computed from the raw input and current opts.",
            kind="calc",
        ),
    }

    __slots__ = ("raw_value", "calc_result")

    def __init__(self, value=2.0, name="example", **kwargs):
        object.__setattr__(self, "raw_value", float(value))
        object.__setattr__(self, "calc_result", None)
        super().__init__(
            opts_type=ExampleOpts,
            name=name,
            **kwargs,
        )
        self.opts.act_finalize(self.opts_defaults)
        self.act_commit(is_reapply_opts=True)

    def _helper_commit_apply_opts_main(self, is_reapply_opts=False, **kwargs):
        with self.opts.act_internal_update():
            for key, value in kwargs.items():
                object.__setattr__(self.opts, key, value)
        object.__setattr__(
            self,
            "calc_result",
            self.raw_value * self.opts.scale + self.opts.offset,
        )


host = ExampleHost(value=3.0)


## What `HostBase` adds to `ClassBase`

A `HostBase` object keeps the identity, attribute conventions, relations, extra attributes, and `show_` inspection vocabulary of `ClassBase`. It adds four major ideas:

1. **A paired options object.** `host.opts` stores configuration parameters that control host behavior.
2. **A commit pipeline.** Public updates are routed through `host.act_commit(...)` rather than treated as isolated assignments.
3. **Reactive recomputation.** Updating host inputs or opts can immediately update dependent `calc_...` or `entity_...` state.
4. **Coordination mechanisms.** Protection, wrappers, synchronization callbacks, and option snapshots help multiple objects stay consistent.

The intended mental model is therefore not “an object plus a bag of settings.” It is closer to “a structured object with managed inputs and managed reactions to those inputs.”


## Start with inspection

Because `HostBase` inherits from `ClassBase`, the same discovery workflow remains useful:

- `show_doc()` asks what the concrete object represents;
- `show_readable_attrs()` asks what can be read;
- `show_modifiable_attrs()` asks what can currently be changed;
- `show_attr_doc(name)` and `show_attr_info(name)` inspect one field;
- `show_relations()` and `show_relation_tree()` inspect object links.

`HostBase` extends the readable/modifiable inspection surface so that fields from the paired opts object appear alongside host fields.


In [ ]:
host.show_doc()
host.show_readable_attrs()
host.show_modifiable_attrs()


### `show_readable_attrs()` on a host

For a `HostBase` object, readable output includes both host-side fields and opts fields. Host-side `raw_` aliases still follow the `ClassBase` convention, while opts fields appear under an `[Opts attributes]` section.

This matters because a concrete host may expose many calculated outputs but only a small set of actual inputs. Reading and modifying remain separate questions.


In [ ]:
host.show_readable_attrs()


### `show_modifiable_attrs()` on a host

`show_modifiable_attrs()` separates the current write surface into categories:

- host attributes;
- opts attributes;
- extra attributes;
- writable host properties.

Protected or wrapped fields are omitted. For an unfamiliar host, this is usually the safest way to determine which inputs are intended for user control.


In [ ]:
host.show_modifiable_attrs()


## The paired `opts` object

Every concrete `HostBase` descendant is paired with an `OptsBase` subclass. The opts object stores configuration variables that belong to the host's behavior rather than to its core identity.

In this example:

```text
host.raw_value   -> the host-side primary input
host.opts.scale  -> a configuration parameter
host.opts.offset -> another configuration parameter
host.calc_result -> dependent output
```

This separation is useful when the same underlying object can be displayed, processed, sampled, or analyzed in different ways without changing what the underlying object fundamentally is.


In [ ]:
print(host.raw_value)
print(host.opts.scale)
print(host.opts.offset)
print(host.calc_result)


### `opts` is reactive

Once an opts object has been finalized and attached to a host, assigning a public opts field is not merely a local dataclass mutation. The assignment is forwarded back to the owning host through `host.act_commit(...)`.

Therefore:


In [ ]:
host.opts.scale = 4.0
print(host.opts.scale)
print(host.calc_result)


Changing `scale` immediately changes the host's dependent result. This “live update” behavior is one of the central design ideas of `HostBase`.

The same opts field may also be assigned through the host:


In [ ]:
host.offset = 5.0
print(host.opts.offset)
print(host.calc_result)


For opts keys, direct assignment on the host is routed through the same commit machinery. In ordinary use, both styles are valid:

```python
host.opts.scale = 2.0
host.scale = 2.0
```

The first style makes it visually explicit that the field belongs to configuration; the second is convenient when working with a host as a single input surface.


## `act_commit()`: the central update operation

`act_commit()` is the explicit batch-update interface for a `HostBase` object. It can receive host inputs, opts inputs, writable properties, and extra attributes in one call.

For example:


In [ ]:
host.act_commit(
    value=10.0,
    scale=0.5,
    offset=1.0,
)

print(host.raw_value)
print(host.opts.scale)
print(host.opts.offset)
print(host.calc_result)


Batch commit is especially useful when several inputs belong to one logical change. Instead of asking the object to react independently to a sequence of separate assignments, the caller can submit the intended update together.

A concrete subclass decides exactly how its opts are applied, but the public contract remains the same: `act_commit()` is the controlled route through which public updates enter the object.


### `raw_` aliases still work

As in `ClassBase`, a host field declared as `raw_value` may normally be addressed through the shorter public alias `value`.

The canonical and alias forms are mutually exclusive within one commit. Passing both would be ambiguous.


In [ ]:
host.act_commit(value=6.0)
print(host.raw_value)
print(host.value)


### Reapplying unchanged opts

Sometimes the opts values themselves have not changed, but a host-side input has changed and all dependent results must be rebuilt from the existing opts.

A subclass can mark a raw/state field with `is_reapply_opts_after_raw=True`, as this example does for `raw_value`. Then changing that field automatically requests opts reapplication.

The same behavior can be requested explicitly:


In [ ]:
host.act_commit(is_reapply_opts=True)
print(host.calc_result)


This distinction is important:

- **opts changed** means new configuration values were supplied;
- **opts reapplication** means the configuration is unchanged, but dependent state should be regenerated from it.


## `OptsBase` as a configuration object

Although most users interact with opts through a host, `OptsBase` also has a small standalone public interface.

`act_asdict()` exports the current public opts payload:


In [ ]:
host.opts.act_asdict()


`str(opts)` gives a compact class identity, while `repr(opts)` gives the full field-by-field configuration:


In [ ]:
print(str(host.opts))
print(repr(host.opts))


### Saving and loading opts as JSON

`OptsBase.act_save_json(path)` serializes the public opts payload. `act_load_json(path)` loads such data back into an opts object.

These methods are useful when option settings should persist across sessions. They are different from the host-side snapshot mechanism described next: JSON persistence writes to disk, while `act_save_opts()` stores an in-memory snapshot on the host.


## Saving opts snapshots on the host

`host.act_save_opts(name)` copies the current opts payload into `host.opts_backup`.


In [ ]:
host.act_save_opts("baseline")
host.show_saved_opts()

print(host.opts_backup["baseline"])


If no name is supplied, `act_save_opts()` creates a timestamp-based key.

A saved snapshot is an ordinary dictionary. To restore one manually, pass it back through the commit pipeline:


In [ ]:
host.act_commit(scale=9.0)
print(host.opts.scale)

host.act_commit(**host.opts_backup["baseline"])
print(host.opts.scale)


## Protection

A host may temporarily prevent selected public inputs from being changed. `HostBase` extends the `ClassBase` protection mechanism to both host attrs and opts attrs.

Use:

- `act_register_protected_attr(...)`;
- `act_unregister_protected_attr(...)`.

The current protected names are readable through `attrs_protected`.


In [ ]:
host.act_register_protected_attr(["value", "scale"])

print(host.attrs_protected)
host.show_modifiable_attrs()

host.act_unregister_protected_attr(["value", "scale"])


`attrs_forbidden` is the union of directly protected fields and wrapped fields:

```python
host.attrs_forbidden == host.attrs_protected | host.attrs_wrapped
```

There are also convenience methods `act_register_protected_opts_all()` and `act_unregister_protected_opts_all()` for protecting or unprotecting the entire opts surface.


## Wrappers and wrapped hosts

`HostBase` can represent a layered object structure in which one host acts as a wrapper around another host. The relations are exposed as:

- `wrapper`: the host that controls this object;
- `wrapped`: the host controlled by this object.

`act_bind_wrapper(wrapper, protected_attrs=...)` establishes both directions of the relation. The wrapped object may mark selected inputs as wrapped-protected so ordinary external assignment cannot bypass the controlling wrapper.


Conceptually:

```text
outer wrapper
    |
    | forwards leftover commit inputs
    v
inner wrapped host
```

When the outer host receives commit keys that it does not consume itself, `HostBase` can forward those remaining keys to the wrapped host. This allows several host layers to present one composed public interface.


The related public operations are:

| Operation | Purpose |
| --- | --- |
| `act_bind_wrapper(wrapper, protected_attrs=...)` | Bind a wrapper and optionally protect wrapped-controlled fields |
| `act_unbind_wrapper()` | Detach the wrapper relationship and clear wrapped protection |
| `act_register_wrapped_attr(...)` | Mark fields as controlled through wrapping |
| `act_unregister_wrapped_attr(...)` | Remove wrapped protection |
| `act_wrapped_update()` | Temporarily suspend wrapped protection inside a managed update |

Most ordinary users encounter these mechanisms indirectly through higher-level visualization or composition classes rather than calling them manually.


## Synchronization callbacks

A host can register post-commit synchronization callbacks. After a successful commit, the host collects the public values that were actually applied and passes them to each registered sync task.

Use:


In [ ]:
events = []


def record_sync(**kwargs):
    events.append(kwargs)


host.act_attach_sync_task("recorder", record_sync)
host.act_commit(value=8.0, offset=2.0)

print(events[-1])

host.act_detach_sync_task("recorder")


This is useful when another object, UI control, cache, or external entity needs to stay consistent with host updates.

A failing sync task is isolated from the rest of the sync batch: `HostBase` logs the failure and skips that callback rather than necessarily invalidating the host commit itself.


### Enriching synchronization or wrapped kwargs

Two more callback registries allow a host to add derived information before data is passed onward:

- `act_attach_enrich_kwargs_sync_task()` / `act_detach_enrich_kwargs_sync_task()` modify the payload sent to synchronization tasks;
- `act_attach_enrich_kwargs_wrapped_task()` / `act_detach_enrich_kwargs_wrapped_task()` modify the kwargs forwarded to a wrapped host.

These are advanced composition tools. They are mainly useful when a concrete host layer needs to translate or augment one update vocabulary before another object receives it.


## Host inputs and outputs

The `ClassBase` prefix convention becomes especially useful for `HostBase`, because the commit pipeline needs a clear distinction between independent inputs and dependent outputs.

A practical rule is:

| Surface | Typical role in a host |
| --- | --- |
| `raw_...` | canonical host-side independent input |
| `state_...` | writable runtime state input |
| writable property | special public input backed by a Python property setter |
| `opts` fields | configuration inputs |
| `calc_...` | calculated output |
| `entity_...` | generated object/entity output |
| `impl_...` | internal execution state |
| extra attrs | user side data, not part of the scientific compute model |
| relations | semantic object links, not automatically commit inputs |

This distinction is what lets `show_modifiable_attrs()` remain meaningful even for complicated hosts.


## A practical workflow for an unfamiliar `HostBase` object

When you receive an object from a part of `Nematics3D` that you have not used before, a useful sequence is:

```python
obj.show_doc()
obj.show_modifiable_attrs()
obj.show_readable_attrs()
obj.show_attr_info("some_field")
repr(obj.opts)
```

Then make updates through either the relevant attribute or `act_commit(...)`.

The important point is that you normally do not need to read the implementation of the host before discovering its public state model. The `show_` interface and paired opts object are intended to expose that model directly.


## Quick reference

| Interface | Purpose |
| --- | --- |
| `host.opts` | Current paired configuration object |
| `host.opts_defaults` | Default opts payload used by the host |
| `host.opts_backup` | In-memory named opts snapshots |
| `host.act_commit(...)` | Apply host/opts updates through the managed pipeline |
| `host.act_save_opts(name=None)` | Save current opts into `opts_backup` |
| `host.show_saved_opts()` | List saved opts snapshots |
| `host.attrs_protected` | Currently directly protected public names |
| `host.attrs_wrapped` | Currently wrapped-protected public names |
| `host.attrs_forbidden` | Union of protected and wrapped names |
| `opts.act_asdict()` | Export current opts as a dictionary |
| `opts.act_save_json(path)` | Save opts to JSON |
| `opts.act_load_json(path)` | Load opts JSON |
| `host.act_attach_sync_task(...)` | Register a post-commit callback |
| `host.act_bind_wrapper(...)` | Establish wrapper/wrapped composition |

The inherited `ClassBase` inspection, naming, extra-attribute, and relation interfaces remain available as well.


## Defining a `HostBase` subclass

**This section is intended for developers. Regular users can safely skip it.**

A concrete host normally provides:

1. an `OptsBase` subclass whose public fields are declared both as dataclass fields and in `__attrs__`;
2. host-side `AttrDef` declarations for its `raw_`, `state_`, `calc_`, `entity_`, relation, or property surfaces;
3. an initialization path that creates the host with the correct `opts_type` and finalizes opts at the appropriate time;
4. `_helper_commit_apply_opts_main()` to translate opts changes into concrete host-side effects.

The central design rule is to keep independent variables and dependent variables distinct. Inputs belong in `raw_...`, `state_...`, writable properties, or opts. Calculated outputs belong in `calc_...` or `entity_...`. Internal execution plumbing belongs in `impl_...`.


### The opts lifecycle

**This section is intended for developers. Regular users can safely skip it.**

Before finalization, an opts object may contain `UNSET` values. `act_finalize()` fills missing fields from supplied defaults and then class-level frozen defaults. Once finalized, the opts object becomes functioning.

After that point, public opts assignment forwards back into the host commit pipeline. A host implementation that needs to update its own opts storage while applying a commit can use `opts.act_internal_update()` to perform that internal mutation without recursively re-entering the host.


### `_helper_commit_apply_opts_main()`

**This section is intended for developers. Regular users can safely skip it.**

`_helper_commit_apply_opts_main()` is the subclass hook that applies opts-domain updates.

A subclass may mutate its opts under `act_internal_update()` and return `None`; `HostBase` will compare the opts payload before and after the hook to determine which opts changed. Alternatively, the hook may return an explicit two-tuple:

```python
(kwargs_left, kwargs_applied_opts)
```

where `kwargs_left` contains values that this host did not consume and may need to be forwarded to a wrapped host.

The public contract should remain stable regardless of which internal hook style a subclass chooses.


### Validation and writable properties

**This section is intended for developers. Regular users can safely skip it.**

Validators for `raw_` and `state_` fields may be registered in their `AttrDef`. Validators for opts fields belong in the opts class's `impl_validators`.

Writable Python properties are a special case. Register the field with `kind="property"` and `is_public_settable=True`, provide an actual property setter, and perform property-specific validation inside that setter. `HostBase` routes the committed value through the setter.


## Summary

`HostBase` turns the structured object vocabulary of `ClassBase` into a reactive object protocol.

The essential user-facing ideas are:

- inspect first with the inherited `show_` interface;
- treat `host.opts` as live configuration rather than a passive settings record;
- use normal public assignment for simple changes and `act_commit()` for explicit or batched updates;
- expect changes in managed inputs to update dependent state;
- use protection, snapshots, synchronization, and wrappers when objects need stronger coordination.

For concrete `Nematics3D` classes, the exact scientific meaning of each option differs, but the host/opts interaction follows this common pattern.
